In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns


# 設置設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(torch.cuda.get_device_name(0))


Using device: cuda
Radeon RX 7900 XTX


In [6]:
import pandas as pd

df = pd.read_csv("Taiwanese/tw_food_101_classes.csv", header=None)

total_classes = df[1].tolist()

print(total_classes)  # 確保讀取正確

['bawan', 'beef_noodles', 'beef_soup', 'bitter_melon_with_salted_eggs', 'braised_napa_cabbage', 'braised_pork_over_rice', 'brown_sugar_cake', 'bubble_tea', 'caozaiguo', 'chicken_mushroom_soup', 'chinese_pickled_cucumber', 'coffin_toast', 'cold_noodles', 'crab_migao', 'deep-fried_chicken_cutlets', 'deep_fried_pork_rib_and_radish_soup', 'dried_shredded_squid', 'egg_pancake_roll', 'eight_treasure_shaved_ice', 'fish_head_casserole', 'fried-spanish_mackerel_thick_soup', 'fried_eel_noodles', 'fried_instant_noodles', 'fried_rice_noodles', 'ginger_duck_stew', 'grilled_corn', 'grilled_taiwanese_sausage', 'hakka_stir-fried', 'hot_sour_soup', 'hung_rui_chen_sandwich', 'intestine_and_oyster_vermicelli', 'iron_egg', 'jelly_of_gravey_and_chicken_feet_skin', 'jerky', 'kung-pao_chicken', 'luwei', 'mango_shaved_ice', 'meat_dumpling_in_chili_oil', 'milkfish_belly_congee', 'mochi', 'mung_bean_smoothie_milk', 'mutton_fried_noodles', 'mutton_hot_pot', 'nabeyaki_egg_noodles', 'night_market_steak', 'nougat',

In [7]:
import os
import warnings
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

class CustomImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, class_map_file=None, transform=None, subset=None, validation_split=0.1, seed=42):
        """
        Args:
            csv_file (str): 影像數據的 CSV，包含 [Index, Label, ImagePath] 或測試集的 [ImagePath]
            root_dir (str): 影像資料的根目錄
            class_map_file (str, optional): Class 對應的 CSV，包含 [Index, ClassName]，可選
            transform (callable, optional): 圖像轉換，如 ToTensor(), Resize() 等
            subset (str, optional): 選擇 'train'、'validation' 或 'test'
            validation_split (float, optional): 設定驗證集比例（0-1 之間）
            seed (int, optional): 隨機種子，確保可重現性
        """
        self.root_dir = root_dir
        self.transform = transform
        self.subset = subset
        self.seed = seed
        self.is_test_set = subset == "test"

        # 根據子集類型讀取CSV
        if self.is_test_set:
            # 測試集CSV只有一列（圖片路徑）
            self.data_frame = pd.read_csv(csv_file, header=None, names=["ImagePath"])
            print(f"測試集數據載入: {len(self.data_frame)} 筆")
        else:
            # 訓練集和驗證集CSV有兩列（標籤和圖片路徑）
            self.data_frame = pd.read_csv(csv_file, header=None, names=["Index", "ImagePath"])
            print(f"訓練/驗證數據載入: {len(self.data_frame)} 筆")

        # 讀取 Class Map（如果有提供）
        if class_map_file:
            class_map_df = pd.read_csv(class_map_file, header=None, names=["Index", "ClassName"])
            self.int_to_class = {row["Index"]: row["ClassName"] for _, row in class_map_df.iterrows()}
            self.class_to_int = {row["ClassName"]: row["Index"] for _, row in class_map_df.iterrows()}
        else:
            self.int_to_class = None
            self.class_to_int = None

        # 如果不是測試集，進行訓練/驗證拆分
        if not self.is_test_set and subset in ["train", "validation"]:
            train_data, val_data = train_test_split(
                self.data_frame, test_size=validation_split, random_state=self.seed
            )
            self.data_frame = train_data if subset == "train" else val_data
            print(f"{'訓練' if subset == 'train' else '驗證'}集數據: {len(self.data_frame)} 筆")

    def __len__(self):
        return len(self.data_frame)
    
    def __getitem__(self, idx):
        # 根據數據集類型獲取正確的圖片路徑列索引
        if self.is_test_set:
            # 測試集只讀取圖片路徑（第一列）
            img_path = os.path.join(self.root_dir, self.data_frame.iloc[idx, 0])
        else:
            # 訓練/驗證集讀取圖片路徑（第二列）
            img_path = os.path.join(self.root_dir, self.data_frame.iloc[idx, 1])

        retries = 0
        max_retries = 5

        while retries < max_retries:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=UserWarning)
                    image = Image.open(img_path)

                # 修正透明度問題
                if image.mode in ["P", "LA", "RGBA"]:
                    image = image.convert("RGB")
                                
                elif image.mode == "CMYK":
                    image = image.convert("RGB")  # 將 CMYK 轉為 RGB

                if self.transform:
                    image = self.transform(image)

                # 測試集沒有標籤
                if self.is_test_set:
                    return image, -1  # 測試集返回None作為標籤
                else:
                    label = int(self.data_frame.iloc[idx, 0])  # 訓練/驗證集使用第一列作為標籤
                    return image, label

            except Exception as e:
                print(f"⚠️ 警告: 無法讀取 {img_path}，錯誤訊息: {e}")
                retries += 1
                # 如果讀取出錯，嘗試重試
                if retries >= max_retries:
                    print(f"跳過圖片: {img_path}")
                    # 返回一個全零的假圖片來避免返回 None
                    dummy_image = Image.new('RGB', (224, 224), (0, 0, 0))  # 創建一個全黑的圖片
                    dummy_label = -1 if not self.is_test_set else None  # 依據數據集類型設定假標籤
                    if self.transform:
                        dummy_image = self.transform(dummy_image)  # 應用相同的 transform
                    return dummy_image, dummy_label  # 返回默認的無效數據

        # 如果超過最大重試次數還是無法讀取圖片，拋出異常
        raise RuntimeError(f"❌ 無法讀取圖片: {img_path}，已達最大重試次數 {max_retries} 次。")

    def get_class_name(self, label_int):
        """ 傳入數字標籤，回傳對應的類別名稱 """
        if self.int_to_class:
            return self.int_to_class.get(label_int, "Unknown")
        return str(label_int)

    def get_class_int(self, class_name):
        """ 傳入類別名稱，回傳對應的數字標籤 """
        if self.class_to_int:
            return self.class_to_int.get(class_name, -1)
        return -1

In [8]:
csv_path = "Taiwanese/tw_food_101/tw_food_101/tw_food_101_train.csv"  # 訓練數據 CSV
test_csv_path = "Taiwanese/tw_food_101/tw_food_101/tw_food_101_test_list.csv"  # 測試數據 CSV
root_dir = "Taiwanese/tw_food_101/tw_food_101/"  # 影像根目錄
class_map_file = "Taiwanese/tw_food_101_classes.csv"  # Class 標籤對應表

In [9]:
# 設定轉換（對於圖像大小、標準化等處理）
# train_transform = transforms.Compose([
#     transforms.RandomHorizontalFlip(p=0.5),  # 增加翻轉機率，通常水平翻轉比較常見
#     transforms.RandomRotation(degrees=10),  # 限制旋轉範圍，避免過度扭曲圖片
#     transforms.AutoAugment(),  # 可以保留，也可以選擇不使用
#     transforms.RandomResizedCrop(256, scale=(0.6,1.0)),  # 使用隨機裁剪來強化模型
#     # transforms.RandAugment(),
#     transforms.ToTensor(),
#     # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ViT 預訓練標準化
# ])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomResizedCrop(384, scale=(0.6, 1.0)),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.8, 1.2), shear=10),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5, interpolation=3),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((384, 384)), # Vit
    transforms.ToTensor(),  # 轉換為 Tensor
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])

# 訓練集 (90%)
train_ds = CustomImageDataset(csv_file=csv_path, root_dir=root_dir, class_map_file=class_map_file, 
                              transform=train_transform, subset='train')

# 驗證集 (10%)
val_ds = CustomImageDataset(csv_file=csv_path, root_dir=root_dir, class_map_file=class_map_file, 
                            transform=val_transform, subset='validation')

test_ds = CustomImageDataset(csv_file=test_csv_path, root_dir=root_dir, class_map_file=class_map_file, 
                             transform=val_transform, subset='test')


# 使用 DataLoader
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory= True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory= True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory= True)




訓練/驗證數據載入: 20372 筆
訓練集數據: 18334 筆
訓練/驗證數據載入: 20372 筆
驗證集數據: 2038 筆
測試集數據載入: 5093 筆


In [10]:
sample = train_ds[0]
print(type(sample))  # 應該是 tuple
print(len(sample))   # 應該是 2（如果包含圖片和標籤）
print(type(sample[0]), sample[0].shape)  # 第一個元素通常是影像
print(type(sample[1]))  # 第二個元素通常是標籤

<class 'tuple'>
2
<class 'torch.Tensor'> torch.Size([3, 384, 384])
<class 'int'>


In [11]:
from timm import create_model

model = create_model("swin_base_patch4_window12_384", pretrained=False, num_classes=len(total_classes))
model.load_state_dict(torch.load("/home/rvl/mingwei/NTUT_Deep_Learning/hw2_weight/revise_vit.pth", map_location=device))
model.to(device)
# 定義損失函數和優化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), weight_decay=0.01, lr=1e-7)  #0.00002122



In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import warnings
import gc

torch.cuda.empty_cache()  # 清空 CUDA Cache
torch.cuda.memory_reserved(0)  # 釋放所有 GPU 記憶體

warnings.filterwarnings("ignore", category=UserWarning, module="PIL")
warnings.filterwarnings("ignore", category=UserWarning, message="Attempting to use hipBLASLt on an unsupported architecture!")

class PseudoLabelTrainer:
    def __init__(
        self, 
        model, 
        train_loader, 
        val_loader, 
        unlabeled_loader, 
        criterion, 
        optimizer, 
        device,
        config=None
    ):
        """
        初始化伪标签训练器
        
        参数:
        - model: 神经网络模型
        - train_loader: 原始训练数据加载器
        - val_loader: 验证数据加载器
        - unlabeled_loader: 未标记数据加载器
        - criterion: 损失函数
        - optimizer: 优化器
        - device: 计算设备
        - config: 配置字典，可包含训练参数
        """
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.unlabeled_loader = unlabeled_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        
        # 默认配置
        self.config = {
            'num_epochs': 30,
            'confidence_threshold': 0.9,
            'pseudo_label_ratio': 0.5,  # 伪标签占总数据比例
            'learning_rate_decay': 0.95,
            'early_stopping_patience': 5
        }
        
        # 更新配置
        if config:
            self.config.update(config)
        
        # 学习率衰减
        self.lr_scheduler = optim.lr_scheduler.ExponentialLR(
            optimizer, 
            gamma=self.config['learning_rate_decay']
        )

    def train_epoch(self, dataloader):
        """
        单个训练轮次
        
        参数:
        - dataloader: 数据加载器
        
        返回:
        - 平均损失和准确率
        """
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        
        progress_bar = tqdm(dataloader, desc="Training", leave=False)
        
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            
            self.optimizer.step()
            
            total_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            progress_bar.set_postfix({
                'loss': loss.item(), 
                'accuracy': correct / total
            })
        
        avg_loss = total_loss / total
        accuracy = correct / total
        
        return avg_loss, accuracy

    def validate(self):
        """
        验证模型性能
        
        返回:
        - 验证损失和准确率
        """
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_loader, desc="Validation", leave=False)
            
            for inputs, labels in progress_bar:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                
                total_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                progress_bar.set_postfix({
                    'loss': loss.item(), 
                    'accuracy': correct / total
                })
        
        avg_loss = total_loss / total
        accuracy = correct / total
        
        return avg_loss, accuracy

    def generate_pseudo_labels(self):
        """
        生成高置信度伪标签
        
        返回:
        - 伪标签图像和标签
        """
        self.model.eval()
        pseudo_images, pseudo_labels = [], []
        confidence_threshold = self.config['confidence_threshold']
        
        with torch.no_grad():
            for images, _ in self.unlabeled_loader:
                if not isinstance(images, torch.Tensor):
                    images = torch.tensor(images)
                images = images.to(self.device)
                outputs = self.model(images)
                confidences, predicted = torch.max(outputs.softmax(dim=1), 1)
                
                # 选择高置信度预测
                high_confidence_mask = confidences >= confidence_threshold
                
                if high_confidence_mask.sum() > 0:
                    pseudo_images.append(images[high_confidence_mask])
                    pseudo_labels.append(predicted[high_confidence_mask])
        
        # 内存优化：限制伪标签数量
        if pseudo_images:
            pseudo_images = torch.cat(pseudo_images)
            pseudo_labels = torch.cat(pseudo_labels)
            
            # 限制伪标签数量
            max_pseudo_labels = int(len(self.train_loader.dataset) * 
                                    self.config['pseudo_label_ratio'])
            
            if len(pseudo_labels) > max_pseudo_labels:
                indices = torch.randperm(len(pseudo_labels))[:max_pseudo_labels]
                pseudo_images = pseudo_images[indices]
                pseudo_labels = pseudo_labels[indices]
            
            return pseudo_images, pseudo_labels
        
        return None, None

    def train(self):
        """
        主训练循环
        """
        best_val_accuracy = 0.0
        early_stopping_counter = 0
        
        for epoch in range(self.config['num_epochs']):
            print(f"\nEpoch {epoch + 1}/{self.config['num_epochs']}")
            
            # 原始训练集训练
            train_loss, train_accuracy = self.train_epoch(self.train_loader)
            val_loss, val_accuracy = self.validate()
            
            print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")
            
            # 学习率衰减
            self.lr_scheduler.step()
            
            # 生成伪标签并扩充训练集
            pseudo_images, pseudo_labels = self.generate_pseudo_labels()
            
            if pseudo_images is not None and pseudo_labels is not None:
                # 创建伪标签数据集
                pseudo_dataset = TensorDataset(
                    pseudo_images.cpu(), 
                    pseudo_labels.cpu()
                )
                
                # 组合数据集
                combined_loader = DataLoader(
                    torch.utils.data.ConcatDataset([
                        self.train_loader.dataset, 
                        pseudo_dataset
                    ]), 
                    batch_size=self.train_loader.batch_size, 
                    shuffle=True,
                    num_workers=1,  # 减少worker数
                    pin_memory=True
                )
                
                # 使用扩充数据集训练
                self.train_epoch(combined_loader)
            
            # 模型保存和提前停止
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                early_stopping_counter = 0
                torch.save(self.model.state_dict(), "best_pseudo_labeling_model.pth")
                print(f"🌟 Model saved with best validation accuracy: {best_val_accuracy:.4f}")
            else:
                early_stopping_counter += 1
            
            # 提前停止
            if early_stopping_counter >= self.config['early_stopping_patience']:
                print("Early stopping triggered.")
                break
            
            # 手动清理内存
            torch.cuda.empty_cache()
            gc.collect()
        
        print(f"\n🎉 Training Complete. Best Validation Accuracy: {best_val_accuracy:.4f}")
        return best_val_accuracy

# 使用示例
def main():
    # 准备你的模型、数据加载器、损失函数和优化器
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 自定义配置（可选）
    config = {
        'num_epochs': 30,
        'confidence_threshold': 0.8,
        'pseudo_label_ratio': 0.1,
        'learning_rate_decay': 0.9,
        'early_stopping_patience': 5
    }
    
    trainer = PseudoLabelTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        unlabeled_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        config=config
    )
    
    best_accuracy = trainer.train()

if __name__ == "__main__":
    main()


Epoch 1/30


OutOfMemoryError: HIP out of memory. Tried to allocate 42.00 MiB. GPU 0 has a total capacity of 23.98 GiB of which 678.00 MiB is free. Of the allocated memory 22.17 GiB is allocated by PyTorch, and 771.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_HIP_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)